In [1]:
# ¿Cómo preparamos los abstracts de Scopus para identificar sus términos distintivos sin que textos vacíos, fórmulas retóricas y avisos de copyright distorsionen la representación?

import pandas as pd

dataframe = pd.read_csv("../data/scopus_abstracts.csv.gz", compression="gzip")
dataframe[["scopus_record_id", "publication_title", "abstract"]].head()

,scopus_record_id,publication_title,abstract
0,2-s2.0-105042759102,Strategic Pathways in Malaysia’s Fintech Secto...,This study examines the relationship between e...
1,2-s2.0-85175999008,"ELI Principles on Blockchain Technology, Smart...",The project started in 2018 following a Use-Ca...
2,2-s2.0-85149119289,Bitland-A Decentralized Commercial Real Estate...,Real estate has always been a significant inve...
3,2-s2.0-84931062324,"The politics of person, property, technology: ...",Global policy organizations have emphasized ca...
4,2-s2.0-85049126196,Planned preventative maintenance and proptech,[No abstract available]


In [2]:
# Los abstracts muy breves suelen ser marcadores de ausencia y no aportan contenido para una representación documento–término.

dataframe["abstract_word_count"] = dataframe["abstract"].str.split().str.len()
dataframe.sort_values("abstract_word_count")[
    ["publication_title", "abstract_word_count", "abstract"]
].head(20)

,publication_title,abstract_word_count,abstract
203,PLANNING FOR THE FUTURE WITH PROPTECH/,3,[No abstract available]
375,"Planning, Land and Housing in the Digital Data...",3,[No abstract available]
39,"Erratum: Intellectual property, technology tra...",3,[No abstract available]
188,Distribution of proptech benefits to stakehold...,3,[No abstract available]
342,Improving global access to new vaccines: Intel...,3,[No abstract available]
106,Digital transformation in real estate marketin...,3,[No abstract available]
219,"Properties, technology, and use possibilities ...",3,[No abstract available]
175,"Intellectual property, technology transfer and...",3,[No abstract available]
25,"Improved superconductor properties, technology...",3,[No abstract available]
328,Trade wars in the TRIPS council: Intellectual ...,3,[No abstract available]


In [3]:
# Se excluyen marcadores conocidos de ausencia y textos demasiado cortos antes de transformar sus palabras.

minimum_abstract_words = 20
missing_abstract_markers = {"[no abstract available]"}

dataframe["has_missing_abstract_marker"] = dataframe["abstract"].str.casefold().isin(
    missing_abstract_markers
)
dataframe["has_sufficient_content"] = (
    ~dataframe["has_missing_abstract_marker"]
    & dataframe["abstract_word_count"].ge(minimum_abstract_words)
)
excluded_abstracts = int((~dataframe["has_sufficient_content"]).sum())
dataframe = dataframe.loc[dataframe["has_sufficient_content"]].copy()
excluded_abstracts, len(dataframe)

(20, 385)

In [4]:
# Los avisos editoriales al final no describen el contenido del artículo y añadirían términos irrelevantes al vocabulario.

copyright_pattern = r"\s*(?:(?:copyright\s*)?©|copyright\b|all rights reserved\b).*?$"
dataframe["has_copyright_notice"] = dataframe["abstract"].str.contains(
    copyright_pattern,
    case=False,
    regex=True,
)
dataframe["has_copyright_notice"].sum()

335

In [5]:
# Se elimina el aviso editorial antes de conservar una versión del abstract orientada al contenido.

dataframe["abstract_without_copyright"] = dataframe["abstract"].str.replace(
    copyright_pattern,
    "",
    case=False,
    regex=True,
)
dataframe[["abstract", "abstract_without_copyright"]].head()

,abstract,abstract_without_copyright
0,This study examines the relationship between e...,This study examines the relationship between e...
1,The project started in 2018 following a Use-Ca...,The project started in 2018 following a Use-Ca...
2,Real estate has always been a significant inve...,Real estate has always been a significant inve...
3,Global policy organizations have emphasized ca...,Global policy organizations have emphasized ca...
5,Coffee is arguably the most popular drink in t...,Coffee is arguably the most popular drink in t...


In [6]:
# Las formulaciones retóricas son frecuentes entre abstracts y no diferencian un tema de investigación de otro.

formulation_patterns = [
    r"\bin this (?:study|paper|article)\b",
    r"\bas a result\b",
    r"\bthe results (?:show|indicate|suggest|reveal)\b",
]

dataframe["normalized_text"] = dataframe["abstract_without_copyright"].str.lower()
for pattern in formulation_patterns:
    dataframe["normalized_text"] = dataframe["normalized_text"].str.replace(
        pattern,
        " ",
        regex=True,
    )

dataframe["normalized_text"] = dataframe["normalized_text"].str.split().str.join(" ")
dataframe[["abstract_without_copyright", "normalized_text"]].head()

,abstract_without_copyright,normalized_text
0,This study examines the relationship between e...,this study examines the relationship between e...
1,The project started in 2018 following a Use-Ca...,the project started in 2018 following a use-ca...
2,Real estate has always been a significant inve...,real estate has always been a significant inve...
3,Global policy organizations have emphasized ca...,global policy organizations have emphasized ca...
5,Coffee is arguably the most popular drink in t...,coffee is arguably the most popular drink in t...


In [7]:
# Se separa cada abstract en tokens para que el texto se convierta en unidades procesables.

from nltk.tokenize import word_tokenize

dataframe["tokens"] = dataframe["normalized_text"].map(word_tokenize)
dataframe[["normalized_text", "tokens"]].head()

,normalized_text,tokens
0,this study examines the relationship between e...,"[this, study, examines, the, relationship, bet..."
1,the project started in 2018 following a use-ca...,"[the, project, started, in, 2018, following, a..."
2,real estate has always been a significant inve...,"[real, estate, has, always, been, a, significa..."
3,global policy organizations have emphasized ca...,"[global, policy, organizations, have, emphasiz..."
5,coffee is arguably the most popular drink in t...,"[coffee, is, arguably, the, most, popular, dri..."


In [8]:
# Se conservan tokens alfabéticos y se hace visible la información que la regla descarta.

dataframe["tokens"] = dataframe["tokens"].map(
    lambda tokens: [token for token in tokens if token.isalpha()]
)
dataframe["processed_text"] = dataframe["tokens"].map(" ".join)
dataframe[["normalized_text", "processed_text"]].head()

,normalized_text,processed_text
0,this study examines the relationship between e...,this study examines the relationship between e...
1,the project started in 2018 following a use-ca...,the project started in following a approach ie...
2,real estate has always been a significant inve...,real estate has always been a significant inve...
3,global policy organizations have emphasized ca...,global policy organizations have emphasized pr...
5,coffee is arguably the most popular drink in t...,coffee is arguably the most popular drink in t...


In [9]:
# ¿Cómo se transforma la lista de tokens en una matriz documento–término apta para analizar el corpus?

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(min_df=2)
document_term_matrix = vectorizer.fit_transform(dataframe["processed_text"])
vocabulary = pd.DataFrame(
    {
        "token": vectorizer.get_feature_names_out(),
        "document_frequency": (document_term_matrix > 0).sum(axis=0).A1,
    }
)
document_term_matrix.shape, vocabulary.head()

((385, 3857),
           token  document_frequency
 0  abbreviation                   2
 1       ability                  15
 2          able                   9
 3         about                  48
 4         above                   6)

In [10]:
# Se guardan abstracts procesados, vocabulario y matriz para conservar una representación reproducible del corpus.

import json
from pathlib import Path
from scipy import sparse

submission_dir = Path("../submission")
dataframe.drop(columns="tokens").to_csv(submission_dir / "tokenized_abstracts.csv", index=False)
vocabulary.to_csv(submission_dir / "vocabulary.csv", index=False)
sparse.save_npz(submission_dir / "document_term_matrix.npz", document_term_matrix)
with (submission_dir / "matrix_metadata.json").open("w", encoding="utf-8") as file:
    json.dump(
        {
            "documents": document_term_matrix.shape[0],
            "terms": document_term_matrix.shape[1],
            "minimum_document_frequency": 2,
            "minimum_abstract_words": minimum_abstract_words,
            "excluded_abstracts": excluded_abstracts,
            "copyright_notices_removed": int(dataframe["has_copyright_notice"].sum()),
        },
        file,
        indent=2,
    )